# Dhan Intraday 15m → BigQuery (thin notebook)

**Git holds only the reusable functions** (the `dhan-pipeline` package).
**This notebook holds every variable** — creds, dates, windows, interval, scrip list.
Edit the VARIABLES cell, then run.

Re-runnable: rows already in BigQuery are skipped (dedup on security_id+timestamp+interval).

In [ ]:
# 1. Install the shared functions from GitHub (fast: skips deps Colab already has)
!pip install -q --force-reinstall --no-deps "git+https://github.com/rajatjain1992/dhan-pipeline.git"

In [ ]:
# 2. Auth: BigQuery via Colab, and Drive for the service-account JSON (needed for gspread)
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
# 3. ===== VARIABLES — edit these, this is the only place values live =====
from dhan_pipeline import Config, run_intraday, latest_intraday_date
from dhan_pipeline import load_scrip_mapping, gspread_client, subset

# --- window/interval knobs (all yours to change) ---
START_DATE  = '2024-01-01'   # first window start (YYYY-MM-DD)
N           = 9             # number of windows
WINDOW_DAYS = 4             # length of each window (was hardcoded 4)
STEP_DAYS   = 7             # gap between window starts (was hardcoded 7)
INTERVAL    = 15           # candle minutes

cfg = Config(
    dhan_client_id    = 'PASTE_CLIENT_ID',
    dhan_access_token = 'PASTE_FRESH_DHAN_TOKEN',
    service_account_file = '/content/drive/MyDrive/Colab Notebooks/rajat-trade-c411eaec7c51.json',
    project_id     = 'rajat-trade',
    dataset_id     = 'stock_data_set',
    intraday_table = 'stock_intraday_prices_dhan',
    sheet_key          = '1aoEgOhQkAAv8b2NqAWtZUYXG41rOal77i0XasevyNtE',
    list_worksheet     = 'my_list',
    negative_worksheet = 'Negative List',
)

In [ ]:
# 3b. Load the scrip list from the Google Sheet, then subset to the 15m set.
#     Needs columns: scrip, security_id, exc_seg, instrument_type.
gc = gspread_client(cfg)
mapping = load_scrip_mapping(cfg, gc)
intraday_15m_scrip_mapping = subset(mapping, 'intraday', ['yes_1'])   # adjust column/value
print(intraday_15m_scrip_mapping[['scrip', 'security_id']].head())

In [ ]:
# 3c. Coverage check — latest candle already in the intraday table
latest_intraday_date(cfg, interval=INTERVAL)

In [ ]:
# 4. Run: fetch all windows -> clean -> dedup vs BQ -> append new candles
result = run_intraday(
    cfg, intraday_15m_scrip_mapping,
    start_date=START_DATE, n=N,
    window_days=WINDOW_DAYS, step_days=STEP_DAYS, interval=INTERVAL,
)
result